In [ ]:
# вспомогательные модули
import pandas as pd
import numpy as np

# предобработка
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler

# проверка моделей
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, precision_score, recall_score, f1_score

# алгоритмы
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVC
from sklearn.ensemble import BaggingRegressor, BaggingClassifier, GradientBoostingRegressor, GradientBoostingClassifier, StackingRegressor, StackingClassifier, RandomForestRegressor, RandomForestClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier

# 1. Изучение примеров
Перед началом выполнения лабораторной работы я изучил предложенные примеры

# 2. Загрузка и подготовка данных

In [33]:
data_regr = pd.read_csv('/content/post_diamonds.csv')
X_regr = data_regr.drop(columns=['price'])
y_regr = data_regr['price']
X_regr_train, X_regr_test, y_regr_train, y_regr_test = train_test_split(X_regr, y_regr, test_size=0.2, random_state=42)

data_clas = pd.read_csv('/content/post_card_transdata.csv')
X_clas = data_clas.drop(columns=['fraud'])
y_clas = data_clas['fraud']
X_clas_train, X_clas_test, y_clas_train, y_clas_test = train_test_split(X_clas, y_clas, test_size=0.2, stratify=y_clas, random_state=42)

In [34]:
scaler = StandardScaler()

X_regr_train_s = scaler.fit_transform(X_regr_train)
X_regr_test_s = scaler.transform(X_regr_test)

rus = RandomUnderSampler(random_state=42, sampling_strategy=0.5)
smote = SMOTE(random_state=42)
X_clas_train_bu, y_clas_train_bu = rus.fit_resample(X_clas_train, y_clas_train)
X_clas_train_b, y_clas_train_b = smote.fit_resample(X_clas_train, y_clas_train)
X_clas_train_bus = scaler.fit_transform(X_clas_train_bu)
X_clas_train_bs = scaler.fit_transform(X_clas_train_b)
X_clas_test_s = scaler.transform(X_clas_test)

# 3. Решение задачи регрессии и классификации

## 3.1.1 DecisionTreeRegressor

In [13]:
des_tree_regr = DecisionTreeRegressor(**{'max_depth': 12, 'min_samples_split': 13, 'min_samples_leaf': 8}, random_state=42)
des_tree_regr.fit(X_regr_train_s, y_regr_train)

DecisionTreeRegressor(ccp_alpha=0.0, criterion='squared_error', max_depth=12,
                      max_features=None, max_leaf_nodes=None,
                      min_impurity_decrease=0.0, min_samples_leaf=8,
                      min_samples_split=13, min_weight_fraction_leaf=0.0,
                      monotonic_cst=None, random_state=42, splitter='best')

In [14]:
y_pred = des_tree_regr.predict(X_regr_test_s)

In [15]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 307.38324115431124
RMSE: 583.6247319504463
R²: 0.9776527845064426
MAPE: 0.08366229574625418


## 3.1.2 DecisionTreeClassifier

In [16]:
des_tree_clas = DecisionTreeClassifier(min_samples_split=5, max_depth=7, criterion='entropy', class_weight='balanced', random_state=42)
des_tree_clas.fit(X_clas_train_bs, y_clas_train_b)

DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced',
                       criterion='entropy', max_depth=7, max_features=None,
                       max_leaf_nodes=None, min_impurity_decrease=0.0,
                       min_samples_leaf=1, min_samples_split=5,
                       min_weight_fraction_leaf=0.0, monotonic_cst=None,
                       random_state=42, splitter='best')

In [17]:
y_pred = des_tree_clas.predict(X_clas_test_s)

In [18]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       1.00      0.98      0.99    182519
           1       0.85      1.00      0.92     17481

    accuracy                           0.98    200000
   macro avg       0.93      0.99      0.96    200000
weighted avg       0.99      0.98      0.99    200000



0.98465

## 3.2.1 BaggingRegressor

In [19]:
ens_bag_regr = BaggingRegressor(**{'n_estimators': 82, 'max_samples': 0.7650220936235242, 'max_features': 0.9943533773759213}, random_state=42, n_jobs=-1)
ens_bag_regr.fit(X_regr_train_s, y_regr_train)

BaggingRegressor(bootstrap=True, bootstrap_features=False, estimator=None,
                 max_features=0.9943533773759213,
                 max_samples=0.7650220936235242, n_estimators=82, n_jobs=-1,
                 oob_score=False, random_state=42, verbose=0, warm_start=False)

In [20]:
y_pred = ens_bag_regr.predict(X_regr_test_s)

In [21]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 264.680945080795
RMSE: 518.2833487725327
R²: 0.9823765652722098
MAPE: 0.06751857865456691


## 3.2.2 BaggingClassifier

In [22]:
ens_bag_clas = BaggingClassifier(**{'n_estimators': 43, 'max_samples': 0.7999412510648061, 'max_features': 0.9521873714773709}, random_state=42, n_jobs=-1)
ens_bag_clas.fit(X_clas_train_bs, y_clas_train_b)

BaggingClassifier(bootstrap=True, bootstrap_features=False, estimator=None,
                  max_features=0.9521873714773709,
                  max_samples=0.7999412510648061, n_estimators=43, n_jobs=-1,
                  oob_score=False, random_state=42, verbose=0,
                  warm_start=False)

In [23]:
y_pred = ens_bag_clas.predict(X_clas_test_s)

In [24]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    182519
           1       0.88      1.00      0.94     17481

    accuracy                           0.99    200000
   macro avg       0.94      0.99      0.97    200000
weighted avg       0.99      0.99      0.99    200000



0.988265

## 3.2.3 GradientBoostingRegressor

In [25]:
gb_regr = GradientBoostingRegressor(**{'n_estimators': 333, 'learning_rate': 0.015750619803704784, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.7276427960501565}, random_state=42)
gb_regr.fit(X_regr_train_s, y_regr_train)

GradientBoostingRegressor(alpha=0.9, ccp_alpha=0.0, criterion='friedman_mse',
                          init=None, learning_rate=0.015750619803704784,
                          loss='squared_error', max_depth=8, max_features=None,
                          max_leaf_nodes=None, min_impurity_decrease=0.0,
                          min_samples_leaf=3, min_samples_split=7,
                          min_weight_fraction_leaf=0.0, n_estimators=333,
                          n_iter_no_change=None, random_state=42,
                          subsample=0.7276427960501565, tol=0.0001,
                          validation_fraction=0.1, verbose=0, warm_start=False)

In [26]:
y_pred = gb_regr.predict(X_regr_test_s)

In [27]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 256.5494699187415
RMSE: 492.5275336206774
R²: 0.9840846179098718
MAPE: 0.0714883126370642


## 3.2.4 GradientBoostingClassifier

In [28]:
gb_clas = GradientBoostingClassifier(**{'n_estimators': 500, 'learning_rate': 0.047491241225455304, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 2, 'subsample': 0.7314298772479082, 'max_features': 'log2'}, random_state=42)
gb_clas.fit(X_clas_train_bus, y_clas_train_bu)

GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                           learning_rate=0.047491241225455304, loss='log_loss',
                           max_depth=4, max_features='log2',
                           max_leaf_nodes=None, min_impurity_decrease=0.0,
                           min_samples_leaf=2, min_samples_split=8,
                           min_weight_fraction_leaf=0.0, n_estimators=500,
                           n_iter_no_change=None, random_state=42,
                           subsample=0.7314298772479082, tol=0.0001,
                           validation_fraction=0.1, verbose=0,
                           warm_start=False)

In [29]:
y_pred = gb_clas.predict(X_clas_test_s)

In [30]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    182519
           1       0.96      0.80      0.87     17481

    accuracy                           0.98    200000
   macro avg       0.97      0.90      0.93    200000
weighted avg       0.98      0.98      0.98    200000



0.979625

## 3.2.5 StackingRegressor

In [35]:
base_models = [
    ('Ridge', Ridge(alpha=25.241424867460356, random_state=42)),
    ('GradientBoosting', GradientBoostingRegressor(**{'n_estimators': 333, 'learning_rate': 0.015750619803704784, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.7276427960501565}, random_state=42)),
    ('RandomTree', RandomForestRegressor(**{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_features': 1.0, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 50, 'verbose': 0, 'warm_start': False}))
]
meta_model = ElasticNet(alpha=0.002183419021890954, l1_ratio=0.6572558012453984, random_state=42)

stack_regr = StackingRegressor(estimators=base_models, final_estimator=meta_model, n_jobs=-1)
stack_regr.fit(X_regr_train_s, y_regr_train)

StackingRegressor(cv=None,
                  estimators=[('Ridge',
                               Ridge(alpha=25.241424867460356, copy_X=True,
                                     fit_intercept=True, max_iter=None,
                                     positive=False, random_state=42,
                                     solver='auto', tol=0.0001)),
                              ('GradientBoosting',
                               GradientBoostingRegressor(alpha=0.9,
                                                         ccp_alpha=0.0,
                                                         criterion='friedman_mse',
                                                         init=None,
                                                         learning_rate=0.015750619803704784,
                                                         loss='squared_error',
                                                         max_dept...
                                                     n_estimators=100,
                                                     n_jobs=-1, oob_score=False,
                                                     random_state=50, verbose=0,
                                                     warm_start=False))],
                  final_estimator=ElasticNet(alpha=0.002183419021890954,
                                             copy_X=True, fit_intercept=True,
                                             l1_ratio=0.6572558012453984,
                                             max_iter=1000, positive=False,
                                             precompute=False, random_state=42,
                                             selection='cyclic', tol=0.0001,
                                             warm_start=False),
                  n_jobs=-1, passthrough=False, verbose=0)

In [36]:
y_pred = stack_regr.predict(X_regr_test_s)

In [37]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 251.85341896748247
RMSE: 490.30688631316826
R²: 0.98422780899724
MAPE: 0.0663179408408268


## 3.2.6 StackingClassifier

In [38]:
_, X_clas_train_bus_svc, _, y_clas_train_bu_svc = train_test_split(X_clas_train_bus, y_clas_train_bu, test_size=0.4, stratify=y_clas_train_bu, random_state=42)

In [39]:
base_models = [
    ('SVC', SVC(gamma=0.1, random_state=42)),
    ('Tree', DecisionTreeClassifier(min_samples_split=5, max_depth=7, criterion='entropy', class_weight='balanced', random_state=42)),
    ('RandomTree', RandomForestClassifier(**{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'gini', 'max_features': 'sqrt', 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 51, 'verbose': 0, 'warm_start': False}))
]
meta_model = LGBMClassifier(**{'n_estimators': 378, 'max_depth': 4, 'learning_rate': 0.045472411633406845, 'num_leaves': 167, 'min_child_samples': 15, 'subsample': 0.787300997249121, 'colsample_bytree': 0.8020043291010864, 'reg_alpha': 0.6629728220339912, 'reg_lambda': 0.12603478242435429}, force_col_wise=True)

stack_clas = StackingClassifier(estimators=base_models, final_estimator=meta_model, n_jobs=-1)
stack_clas.fit(X_clas_train_bus_svc, y_clas_train_bu_svc)

[LightGBM] [Info] Number of positive: 27969, number of negative: 55938
[LightGBM] [Info] Total Bins 327
[LightGBM] [Info] Number of data points in the train set: 83907, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

StackingClassifier(cv=None,
                   estimators=[('SVC',
                                SVC(C=1.0, break_ties=False, cache_size=200,
                                    class_weight=None, coef0=0.0,
                                    decision_function_shape='ovr', degree=3,
                                    gamma=0.1, kernel='rbf', max_iter=-1,
                                    probability=False, random_state=42,
                                    shrinking=True, tol=0.001, verbose=False)),
                               ('Tree',
                                DecisionTreeClassifier(ccp_alpha=0.0,
                                                       class_weight='balanced',
                                                       criterion='entro...
                                                  learning_rate=0.045472411633406845,
                                                  max_depth=4,
                                                  min_child_samples=15,
                                                  min_child_weight=0.001,
                                                  min_split_gain=0.0,
                                                  n_estimators=378, n_jobs=None,
                                                  num_leaves=167,
                                                  objective=None,
                                                  random_state=None,
                                                  reg_alpha=0.6629728220339912,
                                                  reg_lambda=0.12603478242435429,
                                                  subsample=0.787300997249121,
                                                  subsample_for_bin=200000,
                                                  subsample_freq=0),
                   n_jobs=-1, passthrough=False, stack_method='auto',
                   verbose=0)

In [40]:
y_pred = stack_clas.predict(X_clas_test_s)

In [41]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    182519
           1       0.94      0.80      0.87     17481

    accuracy                           0.98    200000
   macro avg       0.96      0.90      0.93    200000
weighted avg       0.98      0.98      0.98    200000



0.978505

## 3.3.1 CatBoost Regression

In [42]:
cat_boost_regr = CatBoostRegressor(**{'iterations': 1120, 'learning_rate': 0.0387592525723667, 'depth': 6, 'l2_leaf_reg': 0.0006357456098930143, 'bagging_temperature': 0.03172310195167838, 'random_strength': 3.7360878294978876, 'grow_policy': 'SymmetricTree'}, task_type='CPU', thread_count=-1, random_state=42)
cat_boost_regr.fit(X_regr_train_s, y_regr_train)

0:	learn: 3876.3185730	total: 32.3ms	remaining: 36.2s
1:	learn: 3744.5004501	total: 42.2ms	remaining: 23.6s
2:	learn: 3622.8470011	total: 49.5ms	remaining: 18.4s
3:	learn: 3509.1698650	total: 56.8ms	remaining: 15.8s
4:	learn: 3399.2064948	total: 64.3ms	remaining: 14.3s
5:	learn: 3293.1042851	total: 71.5ms	remaining: 13.3s
6:	learn: 3191.0808468	total: 80.1ms	remaining: 12.7s
7:	learn: 3091.5989186	total: 91.9ms	remaining: 12.8s
8:	learn: 2999.7389499	total: 112ms	remaining: 13.8s
9:	learn: 2909.9680242	total: 122ms	remaining: 13.6s
10:	learn: 2815.3148049	total: 132ms	remaining: 13.3s
11:	learn: 2730.8512821	total: 141ms	remaining: 13s
12:	learn: 2653.0528805	total: 149ms	remaining: 12.7s
13:	learn: 2577.5955053	total: 156ms	remaining: 12.3s
14:	learn: 2500.5535327	total: 163ms	remaining: 12s
15:	learn: 2427.3150869	total: 171ms	remaining: 11.8s
16:	learn: 2361.0691060	total: 179ms	remaining: 11.6s
17:	learn: 2294.9152652	total: 186ms	remaining: 11.4s
18:	learn: 2226.4756307	total: 194

In [43]:
y_pred = cat_boost_regr.predict(X_regr_test_s)

In [44]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 260.87685572073605
RMSE: 488.90722325750124
R²: 0.9843177291847929
MAPE: 0.07417724345954956


## 3.3.2 CatBoost Classification

In [45]:
cat_boost_clas = CatBoostClassifier(**{'iterations': 1466, 'learning_rate': 0.004896927153059759, 'depth': 7, 'l2_leaf_reg': 0.0015223359176138216, 'border_count': 226, 'grow_policy': 'SymmetricTree'}, task_type='CPU', thread_count=-1, random_state=42)
cat_boost_clas.fit(X_clas_train_bs, y_clas_train_b)

0:	learn: 0.6772416	total: 339ms	remaining: 8m 17s
1:	learn: 0.6608511	total: 653ms	remaining: 7m 57s
2:	learn: 0.6458591	total: 945ms	remaining: 7m 40s
3:	learn: 0.6316009	total: 1.28s	remaining: 7m 46s
4:	learn: 0.6176963	total: 1.6s	remaining: 7m 47s
5:	learn: 0.6024611	total: 1.9s	remaining: 7m 41s
6:	learn: 0.5889067	total: 2.19s	remaining: 7m 36s
7:	learn: 0.5763887	total: 2.49s	remaining: 7m 34s
8:	learn: 0.5625848	total: 2.81s	remaining: 7m 34s
9:	learn: 0.5499552	total: 3.1s	remaining: 7m 31s
10:	learn: 0.5363127	total: 3.41s	remaining: 7m 31s
11:	learn: 0.5242585	total: 3.73s	remaining: 7m 32s
12:	learn: 0.5133988	total: 4.02s	remaining: 7m 29s
13:	learn: 0.5024243	total: 4.31s	remaining: 7m 27s
14:	learn: 0.4912066	total: 4.61s	remaining: 7m 26s
15:	learn: 0.4805218	total: 4.91s	remaining: 7m 25s
16:	learn: 0.4714394	total: 5.21s	remaining: 7m 23s
17:	learn: 0.4623794	total: 5.5s	remaining: 7m 22s
18:	learn: 0.4530581	total: 5.82s	remaining: 7m 23s
19:	learn: 0.4431384	total

In [46]:
y_pred = cat_boost_clas.predict(X_clas_test_s)

In [47]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    182519
           1       0.89      1.00      0.94     17481

    accuracy                           0.99    200000
   macro avg       0.95      0.99      0.97    200000
weighted avg       0.99      0.99      0.99    200000



0.989575

## 3.3.3 XGBoost Regression

In [48]:
xgboost_regr = XGBRegressor(**{'n_estimators': 773, 'max_depth': 7, 'learning_rate': 0.017064549414664634, 'subsample': 0.7747701428698828, 'colsample_bytree': 0.7899746430902906, 'gamma': 0.12319436780270282, 'reg_alpha': 0.009642742572935563, 'reg_lambda': 0.12610503197436898}, random_state=42, n_jobs=-1)
xgboost_regr.fit(X_regr_train_s, y_regr_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7899746430902906, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.12319436780270282,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.017064549414664634,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=773, n_jobs=-1,
             num_parallel_tree=None, objective='reg:squarederror', ...)

In [49]:
y_pred = xgboost_regr.predict(X_regr_test_s)

In [50]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 256.7457918815425
RMSE: 496.9632928025655
R²: 0.9837966555165921
MAPE: 0.07159865212268225


## 3.3.4 XGBoost Classification

In [51]:
xgboost_clas = XGBClassifier(**{'n_estimators': 160, 'max_depth': 5, 'learning_rate': 0.03380924770634301, 'subsample': 0.8674150272975121, 'colsample_bytree': 0.9553110238818484, 'gamma': 0.2949825966699646, 'reg_alpha': 0.014787616897752608, 'reg_lambda': 1.4126794376806364})
xgboost_clas.fit(X_clas_train_bs, y_clas_train_b)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9553110238818484, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.2949825966699646,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03380924770634301,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=160, n_jobs=None,
              num_parallel_tree=None, objective='binary:logistic', ...)

In [52]:
y_pred = xgboost_clas.predict(X_clas_test_s)

In [53]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    182519
           1       0.89      1.00      0.94     17481

    accuracy                           0.99    200000
   macro avg       0.95      0.99      0.97    200000
weighted avg       0.99      0.99      0.99    200000



0.98956

## 3.3.5 LightGBM Regression

In [54]:
light_gbm_regr = LGBMRegressor(**{'n_estimators': 674, 'max_depth': 6, 'learning_rate': 0.014215547294580726, 'num_leaves': 244, 'min_child_samples': 8, 'subsample': 0.959970948373626, 'colsample_bytree': 0.7036459616308876, 'reg_alpha': 0.003675639993320709, 'reg_lambda': 0.2428815144688183}, force_col_wise=True)
light_gbm_regr.fit(X_regr_train_s, y_regr_train)

[LightGBM] [Info] Total Bins 1286
[LightGBM] [Info] Number of data points in the train set: 43035, number of used features: 9
[LightGBM] [Info] Start training from score 3944.883235
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

LGBMRegressor(boosting_type='gbdt', class_weight=None,
              colsample_bytree=0.7036459616308876, force_col_wise=True,
              importance_type='split', learning_rate=0.014215547294580726,
              max_depth=6, min_child_samples=8, min_child_weight=0.001,
              min_split_gain=0.0, n_estimators=674, n_jobs=None, num_leaves=244,
              objective=None, random_state=None, reg_alpha=0.003675639993320709,
              reg_lambda=0.2428815144688183, subsample=0.959970948373626,
              subsample_for_bin=200000, subsample_freq=0)

In [55]:
y_pred = light_gbm_regr.predict(X_regr_test_s)

In [56]:
print(f"MAE: {mean_absolute_error(y_regr_test, y_pred)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_regr_test, y_pred))}")
print(f"R²: {r2_score(y_regr_test, y_pred)}")
print(f"MAPE: {mean_absolute_percentage_error(y_regr_test, y_pred)}")

MAE: 262.7696672879388
RMSE: 496.1804970901989
R²: 0.9838476609698762
MAPE: 0.07648626310468114


## 3.3.6 LightGBM Classification

In [57]:
light_gbm_clas = LGBMClassifier(**{'n_estimators': 378, 'max_depth': 4, 'learning_rate': 0.045472411633406845, 'num_leaves': 167, 'min_child_samples': 15, 'subsample': 0.787300997249121, 'colsample_bytree': 0.8020043291010864, 'reg_alpha': 0.6629728220339912, 'reg_lambda': 0.12603478242435429}, force_col_wise=True)
light_gbm_clas.fit(X_clas_train_bs, y_clas_train_b)

[LightGBM] [Info] Number of positive: 730078, number of negative: 730078
[LightGBM] [Info] Total Bins 1032
[LightGBM] [Info] Number of data points in the train set: 1460156, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

LGBMClassifier(boosting_type='gbdt', class_weight=None,
               colsample_bytree=0.8020043291010864, force_col_wise=True,
               importance_type='split', learning_rate=0.045472411633406845,
               max_depth=4, min_child_samples=15, min_child_weight=0.001,
               min_split_gain=0.0, n_estimators=378, n_jobs=None,
               num_leaves=167, objective=None, random_state=None,
               reg_alpha=0.6629728220339912, reg_lambda=0.12603478242435429,
               subsample=0.787300997249121, subsample_for_bin=200000,
               subsample_freq=0)

In [58]:
y_pred = light_gbm_clas.predict(X_clas_test_s)

In [59]:
print(classification_report(y_clas_test, y_pred))
accuracy_score(y_clas_test, y_pred)

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    182519
           1       0.90      1.00      0.95     17481

    accuracy                           0.99    200000
   macro avg       0.95      0.99      0.97    200000
weighted avg       0.99      0.99      0.99    200000



0.990095

## 3.3.7 ScetchBoost

**ScetchBoost** - часть экспериментальной библиотеки _sketchboost-paper_, которая является "расширенной" версией _py-boost_ под авторством Сбера. Я ознакомился с этим алгоритмом, а так же посмотрел вебинар-презентацию этого алгоритма, включая мотивации его создания и его результаты работы. Использовать на своём устройстве я не могу, так как требуются высокие вычислительные мощности GPU на основе NVIDIA, которыми моё устройство не обладает

## 3.4 Объединение моделей

In [60]:
models_regr = {'DecisionTreeRegressor': des_tree_regr,
               'BaggingRegressor': ens_bag_regr,
               'GradientBoostingRegressor': gb_regr,
               'StackingRegressor': stack_regr,
               'CatBoostRegressor': cat_boost_regr,
               'XGBoostRegressor': xgboost_regr,
               'LightGBMRegressor': light_gbm_regr}

models_clas = {'clas_DecisionTreeClassifier': des_tree_clas,
               'clas_BaggingClassifier': ens_bag_clas,
               'GradientBoostingClassifier': gb_clas,
               'StackingClassifier': stack_clas,
               'CatBoostClassifier': cat_boost_clas,
               'XGBoostClassifier': xgboost_clas,
               'LightGBMClassifier': light_gbm_clas}

# 4. Визуализация

In [61]:
def print_tree_rules_regressor(model, feature_names):
    tree_rules = export_text(
        model,
        feature_names=list(feature_names),
        spacing=3,
        decimals=2
    )
    print("Решающие правила для DecisionTreeRegressor:")
    print(tree_rules)

print_tree_rules_regressor(
    models_regr['DecisionTreeRegressor'],
    feature_names=X_regr.columns
)

Решающие правила для DecisionTreeRegressor:
|--- carat <= 0.41
|   |--- y <= -0.18
|   |   |--- carat <= -0.72
|   |   |   |--- x <= -1.01
|   |   |   |   |--- clarity <= -1.49
|   |   |   |   |   |--- carat <= -1.06
|   |   |   |   |   |   |--- carat <= -1.14
|   |   |   |   |   |   |   |--- cut <= 0.36
|   |   |   |   |   |   |   |   |--- color <= -0.06
|   |   |   |   |   |   |   |   |   |--- x <= -1.54
|   |   |   |   |   |   |   |   |   |   |--- value: [634.45]
|   |   |   |   |   |   |   |   |   |--- x >  -1.54
|   |   |   |   |   |   |   |   |   |   |--- depth <= -0.49
|   |   |   |   |   |   |   |   |   |   |   |--- value: [669.82]
|   |   |   |   |   |   |   |   |   |   |--- depth >  -0.49
|   |   |   |   |   |   |   |   |   |   |   |--- value: [744.00]
|   |   |   |   |   |   |   |   |--- color >  -0.06
|   |   |   |   |   |   |   |   |   |--- color <= 0.53
|   |   |   |   |   |   |   |   |   |   |--- x <= -1.57
|   |   |   |   |   |   |   |   |   |   |   |--- value: [537.62]

In [62]:
def print_tree_rules_classifier(model, feature_names, class_names):
    tree_rules = export_text(
        model,
        feature_names=list(feature_names),
        class_names=class_names,
        spacing=3,
        decimals=2
    )
    print("Решающие правила для DecisionTreeClassifier:")
    print(tree_rules)

print_tree_rules_classifier(
    models_clas['clas_DecisionTreeClassifier'],
    feature_names=X_clas.columns,
    class_names=['0', '1']
)

Решающие правила для DecisionTreeClassifier:
|--- ratio_to_median_purchase_price <= 0.04
|   |--- distance_from_home <= 0.61
|   |   |--- distance_from_last_transaction <= 0.75
|   |   |   |--- ratio_to_median_purchase_price <= -0.35
|   |   |   |   |--- distance_from_home <= 0.27
|   |   |   |   |   |--- ratio_to_median_purchase_price <= -0.40
|   |   |   |   |   |   |--- distance_from_home <= 0.16
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- distance_from_home >  0.16
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- ratio_to_median_purchase_price >  -0.40
|   |   |   |   |   |   |--- distance_from_last_transaction <= 0.62
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- distance_from_last_transaction >  0.62
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |--- distance_from_home >  0.27
|   |   |   |   |   |--- ratio_to_median_purchase_price <= -0.58
|   |   |   |   |   |   |--- distance_from_home <= 0.49
|  

# 5. Оценка качества моделей (создание моделей с помощью PyCaret)

При подключении PyCaret возникли проблемы несовместимости в связи с чем для взаимодействия с библиотекой был использован сервис Google Colab

## Регрессия

In [5]:
import pycaret.regression as pyreg

reg_exp = pyreg.setup(
    data=data_regr,
    target='price',
    session_id=42,
    verbose=True
)

,Description,Value
0,Session id,42
1,Target,price
2,Target type,Regression
3,Original data shape,"(53794, 10)"
4,Transformed data shape,"(53794, 10)"
5,Transformed train set shape,"(37655, 10)"
6,Transformed test set shape,"(16139, 10)"
7,Numeric features,9
8,Preprocess,True
9,Imputation type,simple


In [6]:
best_reg = pyreg.compare_models(sort='R2', n_select=1)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,275.0148,289232.5509,537.3948,0.9821,0.0970,0.0732,8.9820
lightgbm,Light Gradient Boosting Machine,284.2456,302661.6601,549.3548,0.9813,0.1053,0.0807,1.9360
et,Extra Trees Regressor,272.8481,309096.5511,555.5568,0.9808,0.0920,0.0661,10.1420
rf,Random Forest Regressor,274.5714,315351.9473,560.6749,0.9805,0.0911,0.0652,15.6920
xgboost,Extreme Gradient Boosting,283.9446,320930.0556,565.5894,0.9801,0.0967,0.0719,0.3250
gbr,Gradient Boosting Regressor,344.7551,394177.3598,627.1404,0.9756,0.1682,0.1085,3.7490
dt,Decision Tree Regressor,372.2939,594924.9643,770.8738,0.9631,0.1244,0.0864,0.3640
knn,K Neighbors Regressor,477.6299,780944.8517,882.5963,0.9516,0.1735,0.1347,0.2850
ada,AdaBoost Regressor,875.3840,1359093.0550,1165.5022,0.9157,0.3936,0.4074,1.9560
lasso,Lasso Regression,807.2021,1514111.5217,1229.5041,0.9062,0.6499,0.4385,0.2360


Processing:   0%|          | 0/85 [00:00<?, ?it/s]

## Классификация

In [7]:
import pycaret.classification as pyclf

clf_exp = pyclf.setup(
    data=data_clas,
    target='fraud',
    session_id=42,
    verbose=True
)

,Description,Value
0,Session id,42
1,Target,fraud
2,Target type,Binary
3,Original data shape,"(1000000, 9)"
4,Transformed data shape,"(1000000, 9)"
5,Transformed train set shape,"(700000, 9)"
6,Transformed test set shape,"(300000, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


In [8]:
best_clf = pyclf.compare_models(sort='F1', n_select=1)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
dt,Decision Tree Classifier,1.0000,0.9999,0.9998,1.0000,0.9999,0.9999,0.9999,2.8270
rf,Random Forest Classifier,1.0000,1.0000,0.9999,1.0000,0.9999,0.9999,0.9999,96.5280
ada,Ada Boost Classifier,0.9998,1.0000,0.9991,0.9985,0.9988,0.9986,0.9986,40.6690
gbc,Gradient Boosting Classifier,0.9996,1.0000,0.9958,0.9995,0.9977,0.9975,0.9975,193.6410
lightgbm,Light Gradient Boosting Machine,0.9987,1.0000,0.9943,0.9911,0.9927,0.9920,0.9920,17.9820
xgboost,Extreme Gradient Boosting,0.9984,1.0000,0.9915,0.9900,0.9907,0.9898,0.9898,5.5880
et,Extra Trees Classifier,0.9983,1.0000,0.9827,0.9984,0.9905,0.9895,0.9896,59.9220
catboost,CatBoost Classifier,0.9981,1.0000,0.9910,0.9876,0.9893,0.9883,0.9883,127.9870
qda,Quadratic Discriminant Analysis,0.9564,0.9660,0.6489,0.8152,0.7226,0.6993,0.7046,0.8520
lr,Logistic Regression,0.9565,0.9656,0.5760,0.8859,0.6980,0.6757,0.6940,20.6050


Processing:   0%|          | 0/69 [00:00<?, ?it/s]

# 6. Создание таблицы результатов

In [63]:
models_regr_cv = {'DecisionTreeRegressor': DecisionTreeRegressor(**{'max_depth': 12, 'min_samples_split': 13, 'min_samples_leaf': 8}, random_state=42),
                  'BaggingRegressor': BaggingRegressor(**{'n_estimators': 82, 'max_samples': 0.7650220936235242, 'max_features': 0.9943533773759213}, random_state=42, n_jobs=-1),
                  'GradientBoostingRegressor': GradientBoostingRegressor(**{'n_estimators': 333, 'learning_rate': 0.015750619803704784, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.7276427960501565}, random_state=42),
                  'StackingRegressor': StackingRegressor(estimators=[('Ridge', Ridge(alpha=25.241424867460356, random_state=42)), ('GradientBoosting', GradientBoostingRegressor(**{'n_estimators': 333, 'learning_rate': 0.015750619803704784, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 3, 'subsample': 0.7276427960501565}, random_state=42)), ('RandomTree', RandomForestRegressor(**{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_features': 1.0, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 50, 'verbose': 0, 'warm_start': False}))], final_estimator=ElasticNet(alpha=0.002183419021890954, l1_ratio=0.6572558012453984, random_state=42), n_jobs=-1),
                  'CatBoostRegressor': CatBoostRegressor(**{'iterations': 1120, 'learning_rate': 0.0387592525723667, 'depth': 6, 'l2_leaf_reg': 0.0006357456098930143, 'bagging_temperature': 0.03172310195167838, 'random_strength': 3.7360878294978876, 'grow_policy': 'SymmetricTree'}, task_type='CPU', thread_count=-1, random_state=42),
                  'XGBoostRegressor': XGBRegressor(**{'n_estimators': 773, 'max_depth': 7, 'learning_rate': 0.017064549414664634, 'subsample': 0.7747701428698828, 'colsample_bytree': 0.7899746430902906, 'gamma': 0.12319436780270282, 'reg_alpha': 0.009642742572935563, 'reg_lambda': 0.12610503197436898}, random_state=42, n_jobs=-1),
                  'LightGBMRegressor': LGBMRegressor(**{'n_estimators': 674, 'max_depth': 6, 'learning_rate': 0.014215547294580726, 'num_leaves': 244, 'min_child_samples': 8, 'subsample': 0.959970948373626, 'colsample_bytree': 0.7036459616308876, 'reg_alpha': 0.003675639993320709, 'reg_lambda': 0.2428815144688183}, force_col_wise=True)}

models_clas_cv = {'clas_DecisionTreeClassifier': DecisionTreeClassifier(min_samples_split=5, max_depth=7, criterion='entropy', class_weight='balanced', random_state=42),
                  'clas_BaggingClassifier': BaggingClassifier(**{'n_estimators': 43, 'max_samples': 0.7999412510648061, 'max_features': 0.9521873714773709}, random_state=42, n_jobs=-1),
                  'GradientBoostingClassifier': GradientBoostingClassifier(**{'n_estimators': 500, 'learning_rate': 0.047491241225455304, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 2, 'subsample': 0.7314298772479082, 'max_features': 'log2'}, random_state=42),
                  'StackingClassifier': StackingClassifier(estimators=[('SVC', SVC(gamma=0.1, random_state=42)), ('Tree', DecisionTreeClassifier(min_samples_split=5, max_depth=7, criterion='entropy', class_weight='balanced', random_state=42)), ('RandomTree', RandomForestClassifier(**{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'gini', 'max_features': 'sqrt', 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 51, 'verbose': 0, 'warm_start': False}))], final_estimator=LGBMClassifier(**{'n_estimators': 378, 'max_depth': 4, 'learning_rate': 0.045472411633406845, 'num_leaves': 167, 'min_child_samples': 15, 'subsample': 0.787300997249121, 'colsample_bytree': 0.8020043291010864, 'reg_alpha': 0.6629728220339912, 'reg_lambda': 0.12603478242435429}, force_col_wise=True), n_jobs=-1),
                  'CatBoostClassifier': CatBoostClassifier(**{'iterations': 1466, 'learning_rate': 0.004896927153059759, 'depth': 7, 'l2_leaf_reg': 0.0015223359176138216, 'border_count': 226, 'grow_policy': 'SymmetricTree'}, task_type='CPU', thread_count=-1, random_state=42),
                  'XGBoostClassifier': XGBClassifier(**{'n_estimators': 160, 'max_depth': 5, 'learning_rate': 0.03380924770634301, 'subsample': 0.8674150272975121, 'colsample_bytree': 0.9553110238818484, 'gamma': 0.2949825966699646, 'reg_alpha': 0.014787616897752608, 'reg_lambda': 1.4126794376806364}),
                  'LightGBMClassifier': LGBMClassifier(**{'n_estimators': 378, 'max_depth': 4, 'learning_rate': 0.045472411633406845, 'num_leaves': 167, 'min_child_samples': 15, 'subsample': 0.787300997249121, 'colsample_bytree': 0.8020043291010864, 'reg_alpha': 0.6629728220339912, 'reg_lambda': 0.12603478242435429}, force_col_wise=True)}

## 6.1 Регрессия

### Функции

In [64]:
def generate_regression_table1(models_regr, X_train, y_train, X_test, y_test):
    results = []

    for name, model in models_regr.items():
        print(f"Добавление строки для регрессионной модели: {name}")

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_metrics = {
            'Algorithm': name,
            'Train R2': round(r2_score(y_train, y_train_pred), 2),
            'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 4),
            'Train MAE': round(mean_absolute_error(y_train, y_train_pred), 4),
            'Test R2': round(r2_score(y_test, y_test_pred), 2),
            'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 4),
            'Test MAE': round(mean_absolute_error(y_test, y_test_pred), 4)
        }

        results.append(train_metrics)

    return pd.DataFrame(results)

df_regr_table1 = generate_regression_table1(models_regr, X_regr_train_s, y_regr_train, X_regr_test_s, y_regr_test)

Добавление строки для регрессионной модели: DecisionTreeRegressor
Добавление строки для регрессионной модели: BaggingRegressor
Добавление строки для регрессионной модели: GradientBoostingRegressor
Добавление строки для регрессионной модели: StackingRegressor
Добавление строки для регрессионной модели: CatBoostRegressor
Добавление строки для регрессионной модели: XGBoostRegressor
Добавление строки для регрессионной модели: LightGBMRegressor


In [65]:
def generate_regression_table2(models_regr, X, y, X_test_h_o, y_test_h_o):
    results = []
    kf = KFold(n_splits=3, shuffle=True, random_state=42)

    for name, model in models_regr.items():
        print(f"Обработка модели: {name}")

        y_pred_holdout = model.predict(X_test_h_o)

        r2_scores, rmse_scores, mae_scores = [], [], []

        for train_idx, test_idx in kf.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            cv_model = models_regr_cv[name]
            cv_model.fit(X_train, y_train)

            y_pred = cv_model.predict(X_test)
            r2_scores.append(r2_score(y_test, y_pred))
            rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        metrics = {
            'Algorithm': name,
            'Hold-out R2': round(r2_score(y_test_h_o, y_pred_holdout), 2),
            'Hold-out RMSE': round(np.sqrt(mean_squared_error(y_test_h_o, y_pred_holdout)), 4),
            'Hold-out MAE': round(mean_absolute_error(y_test_h_o, y_pred_holdout), 4),
            'K-Fold R2 (mean)': round(np.mean(r2_scores), 2),
            'K-Fold RMSE (mean)': round(np.mean(rmse_scores), 4),
            'K-Fold MAE (mean)': round(np.mean(mae_scores), 4)
        }

        results.append(metrics)

    return pd.DataFrame(results)

df_regr_table2 = generate_regression_table2(models_regr, X_regr, y_regr, X_regr_test_s, y_regr_test)

Выходные данные были обрезаны до нескольких последних строк (5000).
393:	learn: 519.9077897	total: 2.56s	remaining: 4.72s
394:	learn: 519.7348190	total: 2.57s	remaining: 4.72s
395:	learn: 519.3594256	total: 2.58s	remaining: 4.71s
396:	learn: 519.2238726	total: 2.58s	remaining: 4.7s
397:	learn: 518.9175435	total: 2.59s	remaining: 4.7s
398:	learn: 518.5999856	total: 2.6s	remaining: 4.69s
399:	learn: 518.3193457	total: 2.6s	remaining: 4.68s
400:	learn: 518.1569464	total: 2.61s	remaining: 4.67s
401:	learn: 517.9978987	total: 2.61s	remaining: 4.67s
402:	learn: 517.7797889	total: 2.62s	remaining: 4.66s
403:	learn: 517.5641673	total: 2.63s	remaining: 4.65s
404:	learn: 517.4408459	total: 2.63s	remaining: 4.65s
405:	learn: 517.2799050	total: 2.64s	remaining: 4.64s
406:	learn: 517.0484400	total: 2.64s	remaining: 4.63s
407:	learn: 516.8271450	total: 2.65s	remaining: 4.62s
408:	learn: 516.5368659	total: 2.65s	remaining: 4.61s
409:	learn: 516.3247036	total: 2.66s	remaining: 4.61s
410:	learn: 516.09

### Таблицы

In [66]:
df_regr_table1

,Algorithm,Train R2,Train RMSE,Train MAE,Test R2,Test RMSE,Test MAE
0,DecisionTreeRegressor,0.98,497.6098,260.7205,0.98,583.6247,307.3832
1,BaggingRegressor,1.00,263.5555,131.1693,0.98,518.2833,264.6809
2,GradientBoostingRegressor,0.99,421.2451,228.0877,0.98,492.5275,256.5495
3,StackingRegressor,0.99,393.4665,209.8533,0.98,490.3069,251.8534
4,CatBoostRegressor,0.99,453.1789,249.0446,0.98,488.9072,260.8769
5,XGBoostRegressor,0.99,383.3123,218.1048,0.98,496.9633,256.7458
6,LightGBMRegressor,0.99,465.4514,251.6873,0.98,496.1805,262.7697


In [67]:
df_regr_table2

,Algorithm,Hold-out R2,Hold-out RMSE,Hold-out MAE,K-Fold R2 (mean),K-Fold RMSE (mean),K-Fold MAE (mean)
0,DecisionTreeRegressor,0.98,583.6247,307.3832,0.98,620.7680,318.5793
1,BaggingRegressor,0.98,518.2833,264.6809,0.98,556.0614,278.6160
2,GradientBoostingRegressor,0.98,492.5275,256.5495,0.98,525.4151,267.6510
3,StackingRegressor,0.98,490.3069,251.8534,0.92,1100.5427,720.0705
4,CatBoostRegressor,0.98,488.9072,260.8769,0.98,528.1662,273.0047
5,XGBoostRegressor,0.98,496.9633,256.7458,0.98,533.8251,268.3176
6,LightGBMRegressor,0.98,496.1805,262.7697,0.98,531.3557,273.7520


## 6.2 Классификация

### Функции

In [68]:
def generate_classification_table1(models_clas, X_train, y_train, X_test, y_test):
    results = []

    for name, model in models_clas.items():
        print(f"Добавление строки для классификационной модели: {name}")

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_metrics = {
            'Algorithm': name,
            'Train Accuracy': round(accuracy_score(y_train, y_train_pred), 2),
            'Train Precision': round(precision_score(y_train, y_train_pred, average='weighted'), 2),
            'Train Recall': round(recall_score(y_train, y_train_pred, average='weighted'), 2),
            'Train F1': round(f1_score(y_train, y_train_pred, average='weighted'), 2),
            'Test Accuracy': round(accuracy_score(y_test, y_test_pred), 2),
            'Test Precision': round(precision_score(y_test, y_test_pred, average='weighted'), 2),
            'Test Recall': round(recall_score(y_test, y_test_pred, average='weighted'), 2),
            'Test F1': round(f1_score(y_test, y_test_pred, average='weighted'), 2)
        }

        results.append(train_metrics)

    return pd.DataFrame(results)

df_clas_table1 = generate_classification_table1(models_clas, X_clas_train_bus_svc, y_clas_train_bu_svc, X_clas_test_s, y_clas_test)

Добавление строки для классификационной модели: clas_DecisionTreeClassifier
Добавление строки для классификационной модели: clas_BaggingClassifier
Добавление строки для классификационной модели: GradientBoostingClassifier
Добавление строки для классификационной модели: StackingClassifier
Добавление строки для классификационной модели: CatBoostClassifier
Добавление строки для классификационной модели: XGBoostClassifier
Добавление строки для классификационной модели: LightGBMClassifier


In [70]:
def generate_classification_table2(models_clas, X, y, X_test_h_o, y_test_h_o):
    results = []
    kf = KFold(n_splits=3, shuffle=True, random_state=42)

    for name, model in models_clas.items():
        print(f"Обработка классификатора: {name}")

        y_pred_holdout = model.predict(X_test_h_o)

        accuracy_scores, precision_scores = [], []
        recall_scores, f1_scores = [], []

        y_values = y.values if hasattr(y, 'values') else y

        for train_idx, test_idx in kf.split(X):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y_values[train_idx], y_values[test_idx]

            cv_model = models_clas_cv[name]
            cv_model.fit(X_train, y_train)

            y_pred = cv_model.predict(X_test)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            precision_scores.append(precision_score(y_test, y_pred, average='weighted'))
            recall_scores.append(recall_score(y_test, y_pred, average='weighted'))
            f1_scores.append(f1_score(y_test, y_pred, average='weighted'))

        metrics = {
            'Algorithm': name,
            'Hold-out Accuracy': round(accuracy_score(y_test_h_o, y_pred_holdout), 2),
            'Hold-out Precision': round(precision_score(y_test_h_o, y_pred_holdout, average='weighted'), 2),
            'Hold-out Recall': round(recall_score(y_test_h_o, y_pred_holdout, average='weighted'), 2),
            'Hold-out F1': round(f1_score(y_test_h_o, y_pred_holdout, average='weighted'), 2),
            'K-Fold Accuracy (mean)': round(np.mean(accuracy_scores), 2),
            'K-Fold Precision (mean)': round(np.mean(precision_scores), 2),
            'K-Fold Recall (mean)': round(np.mean(recall_scores), 2),
            'K-Fold F1 (mean)': round(np.mean(f1_scores), 2)
        }

        print(metrics)

        results.append(metrics)

    return pd.DataFrame(results)

df_clas_table2 = generate_classification_table2(models_clas, X_clas_train_bus_svc, y_clas_train_bu_svc, X_clas_test_s, y_clas_test)

Выходные данные были обрезаны до нескольких последних строк (5000).
552:	learn: 0.0043241	total: 9.31s	remaining: 15.4s
553:	learn: 0.0043148	total: 9.36s	remaining: 15.4s
554:	learn: 0.0043125	total: 9.39s	remaining: 15.4s
555:	learn: 0.0043058	total: 9.43s	remaining: 15.4s
556:	learn: 0.0042998	total: 9.48s	remaining: 15.5s
557:	learn: 0.0042957	total: 9.53s	remaining: 15.5s
558:	learn: 0.0042932	total: 9.57s	remaining: 15.5s
559:	learn: 0.0042883	total: 9.61s	remaining: 15.5s
560:	learn: 0.0042854	total: 9.65s	remaining: 15.6s
561:	learn: 0.0042803	total: 9.7s	remaining: 15.6s
562:	learn: 0.0042774	total: 9.75s	remaining: 15.6s
563:	learn: 0.0042715	total: 9.78s	remaining: 15.6s
564:	learn: 0.0042655	total: 9.83s	remaining: 15.7s
565:	learn: 0.0042637	total: 9.87s	remaining: 15.7s
566:	learn: 0.0042570	total: 9.91s	remaining: 15.7s
567:	learn: 0.0042526	total: 9.94s	remaining: 15.7s
568:	learn: 0.0042499	total: 9.99s	remaining: 15.7s
569:	learn: 0.0042469	total: 10s	remaining: 15.8s

### Таблицы

In [71]:
df_clas_table1

,Algorithm,Train Accuracy,Train Precision,Train Recall,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,clas_DecisionTreeClassifier,0.96,0.96,0.96,0.96,0.98,0.99,0.98,0.99
1,clas_BaggingClassifier,0.96,0.96,0.96,0.96,0.99,0.99,0.99,0.99
2,GradientBoostingClassifier,1.00,1.00,1.00,1.00,0.98,0.98,0.98,0.98
3,StackingClassifier,1.00,1.00,1.00,1.00,0.98,0.98,0.98,0.98
4,CatBoostClassifier,0.96,0.96,0.96,0.96,0.99,0.99,0.99,0.99
5,XGBoostClassifier,0.89,0.92,0.89,0.90,0.99,0.99,0.99,0.99
6,LightGBMClassifier,0.96,0.96,0.96,0.96,0.99,0.99,0.99,0.99


In [72]:
df_clas_table2

,Algorithm,Hold-out Accuracy,Hold-out Precision,Hold-out Recall,Hold-out F1,K-Fold Accuracy (mean),K-Fold Precision (mean),K-Fold Recall (mean),K-Fold F1 (mean)
0,clas_DecisionTreeClassifier,0.98,0.99,0.98,0.99,1.0,1.0,1.0,1.0
1,clas_BaggingClassifier,0.99,0.99,0.99,0.99,1.0,1.0,1.0,1.0
2,GradientBoostingClassifier,0.98,0.98,0.98,0.98,1.0,1.0,1.0,1.0
3,StackingClassifier,0.98,0.98,0.98,0.98,1.0,1.0,1.0,1.0
4,CatBoostClassifier,0.99,0.99,0.99,0.99,1.0,1.0,1.0,1.0
5,XGBoostClassifier,0.99,0.99,0.99,0.99,1.0,1.0,1.0,1.0
6,LightGBMClassifier,0.99,0.99,0.99,0.99,1.0,1.0,1.0,1.0


# 7. Вывод

**Общие мысли:** деревья решений являются хорошим алгоритмом на уровне баланса скорости и точности, а используя ансамбли моделей и градиентный бустинг можно повысить точность до крайне хороших результатов, однако, необходимо подробно рассмотреть гиперпараметры

**Регрессор:** В результате лабораторной работы был выявлен лучший (наиболее подходящий на моих данных) регрессор - *GradientBoostingRegressor*. Такой вывод следует из анализа метрик каждой модели. Благодаря таблицам явно видно, что этот регрессор обладает наименьшими показателями $RMSE = 525.4151$ и $MAE = 267.6510$ при хорошем показателе $R^2 = 0.98$. <br>
**Однако**, этот классификатор побеждает лишь в точности, в балансе между скоростью и точностью лидирует *XGBoostRegressor* с показателями $RMSE = 533.6009$ и $MAE = 267.9068$ при том же показателе $R^2 = 0.98$

**Классификатор:** В результате лабораторной работы был выявлен так же лучший (наиболее подходящий на моих данных) классификатор - *DecisionTreeClassifier*. Такой вывод следует из анализа метрик каждой модели. Благодаря таблицам явно видно, что метрики всех моделей $Accuracy, Precision, Recall, F1 = 1$ на кросс-валидационной выборке. Из этого можно сделать вывод, что насчёт точности предсказаний модели имеют очень схожий высокий результат, но вот самая быстрая модель - выбранная мной *DecisionTreeClassifier*